In [0]:
"""Practice with JSON (The "Nested" Way)"""

from pyspark.sql.functions import transform, exists, concat, lit

# 1. Create DataFrame directly from JSON strings (no file needed)
json_str = """
{"id": 1, "info": {"tags": ["urgent", "tech", "billing"]}}
{"id": 2, "info": {"tags": ["low", "tech"]}}
"""

# 2. Read JSON directly from string using parallelize
from pyspark.sql import Row
import json
json_lines = [json.loads(line) for line in json_str.strip().split('\n')]
df_json = spark.createDataFrame(json_lines)

# Note: Access the nested array using dot notation inside the function
df_json_applied = df_json.select(
    "id",
    # TRANSFORM: Add a prefix to each tag
    transform("info.tags", lambda t: concat(lit("tag_"), t)).alias("prefixed_tags"),
    
    # EXISTS: Check if 'urgent' is in the nested tags
    exists("info.tags", lambda t: t == "urgent").alias("is_priority")
)

display(df_json_applied)

In [0]:
"""You said
TRANSFORM, FILTER, and EXISTS  Higher-Order Functions -- praticse code with handling csv , json and createdataframe"""

from pyspark.sql.functions import *
from pyspark.sql.types import *

# 1. Setup Data
data = [
    ("Aditya", ["Spark", "Python", "Java"]),
    ("Sita", ["SQL", "Excel"]),
    ("Rahul", ["spark", "pyspark", "MLflow"])
]
schema = "name STRING, skills ARRAY<STRING>"
df = spark.createDataFrame(data, schema)

# 2. Apply Higher-Order Functions
df_result = df.select(
    "name",
    # TRANSFORM: Convert all skills to uppercase
    transform("skills", lambda x: upper(x)).alias("upper_skills"),
    
    # FILTER: Keep only skills that start with 'S' (case-insensitive)
    filter("skills", lambda x: x.like("S%")).alias("s_skills"),
    
    # EXISTS: Boolean flag if they know 'Python'
    exists("skills", lambda x: x == "Python").alias("knows_python")
)

display(df_result)

In [0]:
# Path format: /Volumes/catalog/schema/volume/folder_name
output_volume_path = "/Volumes/main/default/raw_data_volume/exported_users_json"

# Write the data
""" df.write.format("json") \
  .mode("append") \
  .save(output_volume_path)"""
  

df.repartition(1).write.format("json") \
  .mode("overwrite") \
  .save("/Volumes/main/default/raw_data_volume/single_user_file")

print(f"Data successfully written to {output_volume_path}")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS main.default

In [0]:
df.write.mode("overwrite").saveAsTable("database_dev.broze_taxi.user_records")

df.write.mode("overwrite").saveAsTable("main.default.json_data")

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, explode

# 1. Define the Schema (Best Practice)
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("info", StructType([
        StructField("role", StringType(), True),
        StructField("location", StringType(), True)
    ]), True),
    StructField("projects", ArrayType(StringType()), True),
    StructField("active", BooleanType(), True)
])

# 2. Create DataFrame directly from JSON strings (no file needed)
json_data = """[{"id": 1, "name": "Aditya", "info": {"role": "Data Engineer", "location": "Bangalore"}, "projects": ["ETL_Pipeline", "Delta_Migration"], "active": true},
{"id": 2, "name": "Sita", "info": {"role": "ML Engineer", "location": "Hyderabad"}, "projects": ["Model_Training"], "active": true},
{"id": 3, "name": "Rahul", "info": {"role": "Architect", "location": "Mumbai"}, "projects": ["Unity_Catalog", "GenAI_Bot", "Cloud_Security"], "active": false}]"""

# 3. Read JSON directly from string
import json
json_list = json.loads(json_data)
df = spark.createDataFrame(json_list, schema=schema)

display(df)

In [0]:
%sql
    
SHOW CATALOGS

show volumes LIKE 'database_dev.broze_taxi%

In [0]:
%sql
-- Usage
SELECT calculate_discount(100) AS discounted_price;

In [0]:
%sql
CREATE OR REPLACE FUNCTION database_dev.broze_taxi.calculate_discount(price DOUBLE)
RETURNS DOUBLE
RETURN price * 0.9;

In [0]:
{
  "user_id": 101,
  "profile": {
    "first_name": "Aditya",
    "skills": ["Spark", "Python", "SQL"]
  }
}

In [0]:
%sql
create schema if not exists database_dev.broze_taxi